# Treinamento e Avaliação da VGG-16 — Projeto PDI
Este notebook executa o treinamento da CNN **VGG-16** utilizando os otimizadores **SGD** e **Adam** no dataset de células HEp-2, comparando as métricas obtidas diretamente com a **Tabela 3** do artigo:
> *"Preprocessing and Optimizer Impact on Image-Based Autoimmune Disease Diagnosis"*.

### 1. Verificar GPU e Clonar Repositório
Certifique-se de que o ambiente de execução está configurado com GPU (**Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU**).

In [ ]:
!nvidia-smi

In [ ]:
# Clona a branch feat/vgg-16 do projeto (caso não tenha clonado ainda)
import os
if not os.path.exists("Projeto-PDI"):
    !git clone -b feat/vgg-16 https://github.com/orleoncio/Projeto-PDI.git
%cd Projeto-PDI
!git pull origin feat/vgg-16

### 2. Instalar / Verificar Dependências

In [ ]:
!pip install -q scikit-learn pandas pillow torchvision torch

### 3. Validar a Arquitetura da VGG-16
Verifica as 16 camadas treináveis (13 convoluções 3x3 + 3 lineares) e a saída ajustada para 6 classes.

In [ ]:
!python tests/test_vgg16.py

### 4. Treinamento da VGG-16 com SGD (Cenário Principal)
* No artigo, o SGD obteve **98.16% de acurácia** na imagem Original.
* O script treina por 30 épocas, salva os pesos em `results/vgg16_original_sgd.pth` e avalia no conjunto de teste.

In [ ]:
!python main_vgg16.py --optimizer sgd --epochs 30 --batch-size 32

### 5. Treinamento da VGG-16 com Adam
* No artigo, a VGG-16 com Adam colapsou para **19.96% de acurácia** (chutando uma única classe).
* Salva os pesos em `results/vgg16_original_adam.pth`.

In [ ]:
!python main_vgg16.py --optimizer adam --epochs 30 --batch-size 32

### 6. Confrontação Direta com os Resultados do Artigo
Gera um quadro comparativo lado a lado entre os resultados obtidos e a Tabela 3 do artigo científico.

In [ ]:
import torch
import pandas as pd
from data.hep2_dataset import create_dataloaders
from models.vgg16 import create_vgg16
from training.test import test_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, test_loader = create_dataloaders(batch_size=32)

# Valores reportados no artigo (Tabela 3 - VGG-16 Original)
paper_metrics = {
    "sgd": {"accuracy": 0.9816, "f1": 0.9805, "precision": 0.9814, "recall": 0.9798},
    "adam": {"accuracy": 0.1996, "f1": 0.0555, "precision": 0.0333, "recall": 0.1667}
}

comparison_rows = []

for opt in ["sgd", "adam"]:
    path = f"results/vgg16_original_{opt}.pth"
    try:
        model = create_vgg16(num_classes=6).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        print(f"\n>>> Avaliando VGG-16 ({opt.upper()}) <<<")
        res = test_model(model, test_loader, device)
        
        for metric_name in ["accuracy", "f1", "precision", "recall"]:
            val_obtained = res[metric_name] * 100
            val_paper = paper_metrics[opt][metric_name] * 100
            diff = val_obtained - val_paper
            comparison_rows.append({
                "Otimizador": opt.upper(),
                "Métrica": metric_name.capitalize(),
                "Artigo (%)": f"{val_paper:.2f}%",
                "Obtido (%)": f"{val_obtained:.2f}%",
                "Diferença (%)": f"{diff:+.2f}%
            })
    except FileNotFoundError:
        print(f"Arquivo de pesos {path} não encontrado. Execute o treinamento correspondente primeiro.")

if comparison_rows:
    df_comp = pd.DataFrame(comparison_rows)
    print("\n=================== TABELA COMPARATIVA VGG-16 ===================")
    display(df_comp)
